# Data Description
- Author: 詹明翰
- Date first created: 20261106
- Background: 由臺北市2025年死傷交通事故衍生資料篩選主要夜間事故，依已確認規則準備 PCA 與分群輸入。
- Unit of observation: 一列一件事故；沿用上游 `record_id` 作唯一鍵。
- Sample period: 2025年1月至12月。
- Known issues:
  - 主分析排除3筆道路類別、道路型態或事故位置缺失，以及91筆速限為0的夜間事故；因此不代表全部6,795筆夜間事故。
  - 速限0的官方意義尚未確認，本資料不把它當成真實0 km/h；後續敏感度分析可另行處理。
  - 低頻類別未合併。排除紀錄後，道路型態代碼19與事故位置代碼25不再出現在主分析樣本。
  - 這些資料只支援事故案件內的情境分群，不能計算全市道路事故率或推論照明改善的因果效果。
- Definition for each variable:
  - `record_id`: 上游資料內唯一事故識別鍵。
  - `crash_datetime`: 事故日期時間。
  - `speed_limit`: 原始速限代碼中的正值，主分析排除0。
  - `hour_sin`, `hour_cos`: 事故小時的24小時週期轉換。
  - `month_sin`, `month_cos`: 事故月份的12個月週期轉換。
  - `weather_code`, `road_class_code`, `road_form_code`, `crash_position_code`: 原始事故代碼；作為類別變數。
  - `z__*`: 以6,701筆主分析樣本平均數與母體標準差（ddof=0）標準化的數值欄。
  - `變數__代碼`: one-hot 類別欄；一般矩陣為0/1，區塊加權矩陣再除以該原始變數的類別數平方根。
- Data paths:
  - `data/processed/taipei_2025_nighttime_clustering_input.csv`；Role: `analysis`；SHA-256: `02869cf20f9362a6bf3d54ca2ca8e4b75dca983069d220bc527d6dbe9122b0f1`。
  - `data/processed/taipei_2025_nighttime_clustering_matrix_unweighted.csv`；Role: `analysis`；SHA-256: `6a6d92299d333cce4dcc9b1f425faa28e0d8a20ba2b76dc7221ecc1d85328ab6`。
  - `data/processed/taipei_2025_nighttime_clustering_matrix_block_weighted.csv`；Role: `analysis`；SHA-256: `88173998c8c597af0ce653e1b6764b03ed2582cdb961c3975de12079ab304e37`。
- Source data: `data/processed/taipei_2025_descriptive.csv`。
- Processing code: `data/code/prepare_nighttime_clustering_data.ipynb`。
- Verification status: 三份檔案已重新讀取並通過下列列數、欄位、唯一鍵、缺失、尺度、one-hot及區塊加權檢查。
- Downstream analysis inputs: 必須使用上述精確路徑與雜湊；PCA／分群比較一般矩陣與區塊加權矩陣。


In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'templates').is_dir():
            return candidate
    raise FileNotFoundError('請從含 AGENTS.md 與 templates/ 的學生專案內開啟本 Notebook。')


def project_file(relative_path, area='processed'):
    rel = Path(relative_path)
    if rel.is_absolute() or '..' in rel.parts:
        raise ValueError(f'必須使用明確的專案相對路徑：{relative_path}')
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / 'data' / area).resolve()
    if not path.is_relative_to(allowed) or not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


PROJECT_ROOT = find_project_root()


## Change for Your Own Data

本文件只讀取下方三個精確路徑，並在載入前核對各自 SHA-256。任一檔案缺少或版本改變時停止，不以其他 CSV、修改日期或 glob 順序替代。


In [2]:
FILES = {
    'semantic': 'data/processed/taipei_2025_nighttime_clustering_input.csv',
    'unweighted': 'data/processed/taipei_2025_nighttime_clustering_matrix_unweighted.csv',
    'block_weighted': 'data/processed/taipei_2025_nighttime_clustering_matrix_block_weighted.csv',
}
EXPECTED_SHA256 = {
    'semantic': '02869cf20f9362a6bf3d54ca2ca8e4b75dca983069d220bc527d6dbe9122b0f1',
    'unweighted': '6a6d92299d333cce4dcc9b1f425faa28e0d8a20ba2b76dc7221ecc1d85328ab6',
    'block_weighted': '88173998c8c597af0ce653e1b6764b03ed2582cdb961c3975de12079ab304e37',
}
EXPECTED_ROWS = 6701
SEMANTIC_COLUMNS = [
    'record_id', 'crash_datetime', 'speed_limit', 'hour_sin', 'hour_cos',
    'month_sin', 'month_cos', 'weather_code', 'road_class_code',
    'road_form_code', 'crash_position_code',
]
NUMERIC_COLUMNS = ['z__speed_limit', 'z__hour_sin', 'z__hour_cos', 'z__month_sin', 'z__month_cos']
LEVELS = {
    'weather_code': [6, 7, 8, 9],
    'road_class_code': [4, 5, 6, 7, 8],
    'road_form_code': [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 18, 22],
    'crash_position_code': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19, 20, 22, 23, 24],
}
ONE_HOT_COLUMNS = []
for field, levels in LEVELS.items():
    ONE_HOT_COLUMNS.extend(sorted([f'{field}__{level}' for level in levels]))
MATRIX_COLUMNS = ['record_id'] + NUMERIC_COLUMNS + ONE_HOT_COLUMNS
assert len(ONE_HOT_COLUMNS) == 44 and len(MATRIX_COLUMNS) == 50

paths = {key: project_file(path) for key, path in FILES.items()}
for key, path in paths.items():
    actual = sha256_file(path)
    if actual != EXPECTED_SHA256[key]:
        raise ValueError(f'{key} 版本改變：請先辨識新版本。')


## Summary


In [3]:
frames = {key: pd.read_csv(path, encoding='utf-8-sig') for key, path in paths.items()}
summary = pd.DataFrame([
    {
        'key': key, 'data_path': FILES[key], 'rows': len(frame),
        'columns': len(frame.columns), 'missing_cells': int(frame.isna().sum().sum()),
        'record_id_unique': frame['record_id'].is_unique, 'sha256': sha256_file(paths[key]),
    }
    for key, frame in frames.items()
])
display(summary)


,key,data_path,rows,columns,missing_cells,record_id_unique,sha256
0,semantic,data/processed/taipei_2025_nighttime_clusterin...,6701,11,0,True,02869cf20f9362a6bf3d54ca2ca8e4b75dca983069d220...
1,unweighted,data/processed/taipei_2025_nighttime_clusterin...,6701,50,0,True,6a6d92299d333cce4dcc9b1f425faa28e0d8a20ba2b76d...
2,block_weighted,data/processed/taipei_2025_nighttime_clusterin...,6701,50,0,True,88173998c8c597af0ce653e1b6764b03ed2582cdb961c3...


### Sample


In [4]:
display(frames['semantic'].head(10))
display(frames['unweighted'].head(5))
display(frames['block_weighted'].head(5))


,record_id,crash_datetime,speed_limit,hour_sin,hour_cos,month_sin,month_cos,weather_code,road_class_code,road_form_code,crash_position_code
0,1,2025-01-01 00:18:00,40,0.000000,1.000000e+00,0.0,1.0,8,5,14,9
1,2,2025-01-01 00:31:00,50,0.000000,1.000000e+00,0.0,1.0,7,5,14,9
2,3,2025-01-01 00:35:00,50,0.000000,1.000000e+00,0.0,1.0,7,5,5,1
3,4,2025-01-01 00:41:00,50,0.000000,1.000000e+00,0.0,1.0,7,5,14,7
4,5,2025-01-01 00:57:00,50,0.000000,1.000000e+00,0.0,1.0,7,5,3,1
5,6,2025-01-01 01:34:00,30,0.258819,9.659258e-01,0.0,1.0,7,5,3,1
6,7,2025-01-01 02:15:00,50,0.500000,8.660254e-01,0.0,1.0,7,5,14,9
7,8,2025-01-01 06:26:00,30,1.000000,6.123234e-17,0.0,1.0,8,5,12,9
8,9,2025-01-01 06:29:00,50,1.000000,6.123234e-17,0.0,1.0,8,5,3,2
9,37,2025-01-01 17:40:00,50,-0.965926,-2.588190e-01,0.0,1.0,6,5,5,1


,record_id,z__speed_limit,z__hour_sin,z__hour_cos,z__month_sin,z__month_cos,weather_code__6,weather_code__7,weather_code__8,weather_code__9,...,crash_position_code__22,crash_position_code__23,crash_position_code__24,crash_position_code__3,crash_position_code__4,crash_position_code__5,crash_position_code__6,crash_position_code__7,crash_position_code__8,crash_position_code__9
0,1,-0.652266,1.230776,1.451848,0.098954,1.251516,0,0,1,0,...,0,0,0,0,0,0,0,0,0,1
1,2,0.558324,1.230776,1.451848,0.098954,1.251516,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1
2,3,0.558324,1.230776,1.451848,0.098954,1.251516,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0.558324,1.230776,1.451848,0.098954,1.251516,0,1,0,0,...,0,0,0,0,0,0,0,1,0,0
4,5,0.558324,1.230776,1.451848,0.098954,1.251516,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


,record_id,z__speed_limit,z__hour_sin,z__hour_cos,z__month_sin,z__month_cos,weather_code__6,weather_code__7,weather_code__8,weather_code__9,...,crash_position_code__22,crash_position_code__23,crash_position_code__24,crash_position_code__3,crash_position_code__4,crash_position_code__5,crash_position_code__6,crash_position_code__7,crash_position_code__8,crash_position_code__9
0,1,-0.652266,1.230776,1.451848,0.098954,1.251516,0.0,0.0,0.5,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.213201
1,2,0.558324,1.230776,1.451848,0.098954,1.251516,0.0,0.5,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.213201
2,3,0.558324,1.230776,1.451848,0.098954,1.251516,0.0,0.5,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000
3,4,0.558324,1.230776,1.451848,0.098954,1.251516,0.0,0.5,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.213201,0.0,0.000000
4,5,0.558324,1.230776,1.451848,0.098954,1.251516,0.0,0.5,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000


### Summary Stats


In [5]:
display(frames['semantic'].describe(include='all').T)
category_summary = pd.DataFrame([
    {'field': field, 'observed_levels': sorted(frames['semantic'][field].unique().tolist()),
     'levels_n': frames['semantic'][field].nunique()}
    for field in LEVELS
])
display(category_summary)
numeric_scaling_check = pd.DataFrame({
    'mean': frames['unweighted'][NUMERIC_COLUMNS].mean(),
    'std_ddof0': frames['unweighted'][NUMERIC_COLUMNS].std(ddof=0),
})
display(numeric_scaling_check)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
record_id,6701.0,NaN,NaN,NaN,11658.830324,7137.961651,1.0,5142.0,11872.0,18394.0,22762.0
crash_datetime,6701,6480,2025-09-26 19:00:00,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
speed_limit,6701.0,NaN,NaN,NaN,45.388002,8.26105,20.0,40.0,50.0,50.0,80.0
hour_sin,6701.0,NaN,NaN,NaN,-0.630601,0.512398,-1.0,-0.965926,-0.866025,-0.5,1.0
hour_cos,6701.0,NaN,NaN,NaN,0.432456,0.390941,-0.258819,-0.0,0.5,0.866025,1.0
month_sin,6701.0,NaN,NaN,NaN,-0.070506,0.712567,-1.0,-0.866025,0.0,0.5,1.0
month_cos,6701.0,NaN,NaN,NaN,0.145494,0.682828,-1.0,-0.5,0.5,0.866025,1.0
weather_code,6701.0,NaN,NaN,NaN,7.269064,0.750908,6.0,7.0,7.0,8.0,9.0
road_class_code,6701.0,NaN,NaN,NaN,5.069542,0.410241,4.0,5.0,5.0,5.0,8.0
road_form_code,6701.0,NaN,NaN,NaN,7.441725,5.037242,3.0,4.0,4.0,14.0,22.0


,field,observed_levels,levels_n
0,weather_code,"[6, 7, 8, 9]",4
1,road_class_code,"[4, 5, 6, 7, 8]",5
2,road_form_code,"[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 18, 22]",13
3,crash_position_code,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15...",22


,mean,std_ddof0
z__speed_limit,1.272424e-16,1.0
z__hour_sin,1.399666e-16,1.0
z__hour_cos,-6.786261e-17,1.0
z__month_sin,1.357252e-16,1.0
z__month_cos,3.393131e-17,1.0


### Assertions


In [6]:
semantic = frames['semantic']
unweighted = frames['unweighted']
block_weighted = frames['block_weighted']

assert list(semantic.columns) == SEMANTIC_COLUMNS
assert list(unweighted.columns) == MATRIX_COLUMNS
assert list(block_weighted.columns) == MATRIX_COLUMNS
for frame in frames.values():
    assert len(frame) == EXPECTED_ROWS
    assert frame['record_id'].is_unique and frame['record_id'].notna().all()
    assert not frame.isna().any().any()
assert semantic['record_id'].tolist() == unweighted['record_id'].tolist()
assert semantic['record_id'].tolist() == block_weighted['record_id'].tolist()
assert semantic['speed_limit'].gt(0).all()
assert pd.to_datetime(semantic['crash_datetime'], errors='coerce').notna().all()
for field, levels in LEVELS.items():
    assert set(semantic[field].unique()) == set(levels)

for column in NUMERIC_COLUMNS:
    assert abs(unweighted[column].mean()) < 1e-12
    assert abs(unweighted[column].std(ddof=0) - 1) < 1e-12
    np.testing.assert_allclose(unweighted[column], block_weighted[column], rtol=0, atol=1e-14)

for field, levels in LEVELS.items():
    columns = sorted([f'{field}__{level}' for level in levels])
    np.testing.assert_allclose(unweighted[columns].sum(axis=1), 1.0, rtol=0, atol=1e-14)
    np.testing.assert_allclose(
        block_weighted[columns].sum(axis=1), 1 / np.sqrt(len(levels)), rtol=0, atol=1e-14
    )
    np.testing.assert_allclose(
        block_weighted[columns], unweighted[columns] / np.sqrt(len(levels)), rtol=0, atol=1e-14
    )

for frame in [unweighted, block_weighted]:
    assert np.isfinite(frame.drop(columns='record_id').to_numpy()).all()
    assert not any(token in column.lower() for column in frame.columns
                   for token in ['light', 'fatal', 'death', 'injury', 'district', 'longitude', 'latitude'])
for key, path in paths.items():
    assert sha256_file(path) == EXPECTED_SHA256[key]

print({
    'status': '三份 processed data 的設定檢查通過',
    'rows_each': EXPECTED_ROWS, 'semantic_columns': len(SEMANTIC_COLUMNS),
    'matrix_columns_each': len(MATRIX_COLUMNS), 'one_hot_columns': len(ONE_HOT_COLUMNS),
    'remaining_known_issues': ['速限0的官方意義尚未確認', '主分析結果需與兩種矩陣及後續敏感度比較'],
})


{'status': '三份 processed data 的設定檢查通過', 'rows_each': 6701, 'semantic_columns': 11, 'matrix_columns_each': 50, 'one_hot_columns': 44, 'remaining_known_issues': ['速限0的官方意義尚未確認', '主分析結果需與兩種矩陣及後續敏感度比較']}
